# 第 1 周 Day 2：Token、Embedding 与位置 — Notebook 作业

[← Week 01 / Day 01](day-01.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-02.md) · [Goal 进度](../../PROGRESS.md) · [Week 01 / Day 03 →](day-03.ipynb)

> 状态：**未提交**。直接编辑各个 Markdown/Code 单元格；“教练验收区”不要预填。


## Goal

预计 75–100 分钟。理解文本 token、图像 patch token、embedding 和位置编码，能从原始输入推导 `[B,N,d_model]`，并用小张量验证 reshape。

### 我的目标复述

【双击此 Markdown 单元格，用自己的话填写今日目标及其在 VLA 中的作用。】


## Setup

| 字段 | 我的记录 |
|---|---|
| 实际投入时间 | 【填写】 |
| 完成日期 | 【填写】 |
| Python / PyTorch | 【填写；纯理论日写“不适用”】 |
| CPU / GPU / 仿真器 | 【填写】 |
| 资源等级 | 【L0 / L1 / L2】 |
| 产物路径 | 【填写】 |

生成时环境检查（2026-09-01）：当前可见 Python 未检测到 Jupyter、ipykernel、nbformat、PyTorch 或 NumPy。本 Notebook 已做结构验证，但在该环境中尚未执行。


In [ ]:
# 可选：Notebook 环境可用后运行此单元，记录基础环境。
import platform
import sys

print("python:", sys.version)
print("platform:", platform.platform())


## Context：知识点及其在 VLA 中的作用

Transformer 接收的是 token 序列而非原始字符串或像素。文本 tokenizer 产生离散 id，embedding 查表得到连续向量；图像被切成 patch 后经线性投影得到视觉 token。位置编码让模型区分“哪个词/哪个图块在哪里”。VLA 的多模态融合首先要求这些序列具有可兼容的特征维度。


## Concepts：概念、公式、形状与数据流

图像 `H×W`、方形 patch 边长 `P` 时：

$$
N_{img}=\frac{H}{P}\frac{W}{P}
$$

当 `H=W=224,P=16`，得到 `14×14=196` 个 patch。每个 patch 原始长度为 `3×16×16=768`，投影到 $d_{\mathrm{model}}=256$：

```text
image [B,3,224,224]
 -> patches [B,196,768]
 -> image_tokens [B,196,256]

token_ids [B,16]
 -> embedding lookup [B,16,256]
 -> + positional encoding [B,16,256]
```

若使用 `[CLS]` token，序列长度会从 196 变 197；必须在接口说明中写清。

### 我的笔记：RGB patch 与 d_model

一个 RGB patch 保留同一空间区域的三个通道，而不是把 R、G、B 分成三个 patch。若通道数为 $C$、patch 边长为 $P$，单个 patch 的形状为 `[C,P,P]`，展平后长度为 $CP^2$。

本例中，一个 patch 是 `[3,16,16]`，展平后为 $3\times16\times16=768$；`196` 来自 $14\times14$ 个空间 patch。随后共享线性层将每个 768 维 patch 向量投影为 256 维：$z=xW+b$，其中 $W\in\mathbb{R}^{768\times256}$。

因此，`d_model=256` 是 Transformer 内部统一的隐藏维度，便于与文本 token 融合；它并不是由 $16\times16=256$ 推导而来，只是数值恰好相同。若是灰度图，原始 patch 长度才是 $1\times16\times16=256$。


## Learning Steps

1. 手算三组 `H,W,P` 的 patch 数量，确认整除条件。
2. 比较 token id `[B,L]` 与 embedding `[B,L,D]` 的 dtype 和含义。
3. 在纸上标记二维 patch `(row,col)` 展平到一维序号的规则。
4. 用 PyTorch 或 NumPy 创建 `[2,3,32,32]`，按 `P=8` 重排为 `[2,16,192]`。
5. 为文本与图像各加入同形状位置向量，检查形状不变。


In [2]:
import torch

torch.manual_seed(7)

B, C, H, W = 2, 3, 32, 32
P = 8
L = 6
D = 32

def patchify(x, patch_size):
    """[B,C,H,W] -> [B,N,C*P*P]"""
    B, C, H, W = x.shape
    assert H % patch_size == 0 and W % patch_size == 0

    h_patches = H // patch_size
    w_patches = W // patch_size

    return (
        x.reshape(B, C, h_patches, patch_size, w_patches, patch_size)
         .permute(0, 2, 4, 1, 3, 5)
         .reshape(B, h_patches * w_patches, C * patch_size * patch_size)
    )

# 模拟两张 RGB 图像：[2, 3, 32, 32]
x = torch.arange(B * C * H * W, dtype=torch.float32)
x = x.reshape(B, C, H, W) / 255.0

# 图像 patch 与投影后的视觉 token
patches = patchify(x, P)
image_projection = torch.nn.Linear(C * P * P, D)
image_tokens = image_projection(patches)

# 文本 token id 与 embedding
token_ids = torch.tensor(
    [[1, 2, 3, 4, 0, 0],
     [5, 6, 7, 8, 9, 0]],
    dtype=torch.long,
)
text_embedding = torch.nn.Embedding(num_embeddings=100, embedding_dim=D)
text_tokens = text_embedding(token_ids)

# 可与 token 相加的位置编码
image_pos = torch.zeros(1, patches.shape[1], D)
text_pos = torch.zeros(1, token_ids.shape[1], D)

print("patches:", patches.shape)
print("image_tokens:", image_tokens.shape)
print("text_tokens:", text_tokens.shape)

patches: torch.Size([2, 16, 192])
image_tokens: torch.Size([2, 16, 32])
text_tokens: torch.Size([2, 6, 32])


## Steps：必做作业

### 课程题目

实现或清晰推演 `patchify(x, patch_size)`：输入 `[2,3,32,32]`，输出 `[2,16,192]`。再构造文本 id `[2,6]`、embedding 后 `[2,6,32]`，列出图像投影为 32 维后的融合前形状。解释交换两个位置向量为什么会改变模型对顺序/空间的判断。

下面每道题都有独立作答单元。文字、表格、公式或 Mermaid 写在 Markdown 单元；可运行代码写在后面的 Code 单元。


### 第 1 题作答

图像 patchify

输入图像的形状是 [B,C,H,W]=[2,3,32,32]。设 patch 边长为 P=8，则高和宽各可分成 $32/8=4$ 个 patch，因此每张图像共有：

$$
N=(H/P)(W/P)=4\times4=16
$$

个视觉 token。

每个 RGB patch 的原始形状为 [3,8,8]，展平后长度为：

$$
C P^2=3\times8\times8=192
$$

因此 patchify(x, 8) 的输出形状为 [2,16,192]。

实现时不能直接把 [B,C,H,W] reshape 成 [B,N,192]，否则空间位置与 RGB 通道的排列可能被打乱。正确流程是先拆开 patch 网格与 patch 内部维度，再调整维度顺序，最后展平：

[B,C,H,W]
→ [B,C,4,8,4,8]
→ [B,4,4,C,8,8]
→ [B,16,192]

### 第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 可运行代码 / 实验区

纯理论日可以保留为空；代码日请将实现拆成短小单元，并保留关键输出。


In [4]:
# 在此编写或运行当天代码。
# 建议先写清输入 shape、dtype、设备和随机种子。


## Checks：输入、预期输出与验证

- 输入：随机浮点图像 `[2,3,32,32]`，`P=8`；整数 token id `[2,6]`；`d_model=32`。
- 预期输出：patch `[2,16,192]`、视觉 token `[2,16,32]`、文本 token `[2,6,32]`。随机数具体值不作要求。
- 验证：`16*192 == 3*32*32`；重排前后元素总数相同；token id dtype 为整数；位置相加前后形状一致。使用 `assert` 保存日志。

低资源替代：不用框架，画 `8×8` 单通道图按 `P=4` 分成 4 块，写出每块的展平顺序。

### 我的验证记录

| 检查项 | 实际结果 | 是否符合 | 证据 |
|---|---|---|---|
| 输入 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| 输出 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| dtype、范围和单位 | 【填写】 | 【填写】 | 【填写】 |
| 正向测试 | 【填写】 | 【填写】 | 【填写】 |
| 负向测试 / 错误注入 | 【填写】 | 【填写】 | 【填写】 |
| 指标分子 / 分母 / seed | 【填写】 | 【填写】 | 【填写】 |

> 尚未运行的内容必须标为“预期结果”，不能作为实际证据。


In [3]:
# Checks：验证 shape、dtype、数值有效性与位置编码

import torch

# 1. 预期 shape
assert patches.shape == (2, 16, 192), f"patches 实际为 {patches.shape}"
assert image_tokens.shape == (2, 16, 32), f"image_tokens 实际为 {image_tokens.shape}"
assert text_tokens.shape == (2, 6, 32), f"text_tokens 实际为 {text_tokens.shape}"

# 2. patch 元素总数守恒
# 16 个 patch × 每个 3×8×8=192 个元素 = 原图 3×32×32 个元素
assert 16 * 192 == 3 * 32 * 32

# 3. 文本 token id 必须是整数；embedding 是浮点数
assert token_ids.dtype == torch.long, f"token_ids dtype 为 {token_ids.dtype}"
assert image_tokens.dtype.is_floating_point
assert text_tokens.dtype.is_floating_point

# 4. 位置编码必须能与 token 序列相加
assert image_pos.shape[-2:] == image_tokens.shape[-2:]
assert text_pos.shape[-2:] == text_tokens.shape[-2:]

# 5. 结果中不应包含 NaN 或 Inf
assert torch.isfinite(image_tokens).all()
assert torch.isfinite(text_tokens).all()

# 6. 负向测试：长度错误的位置编码不能匹配图像 token
bad_image_pos = torch.zeros(1, 15, 32)
assert bad_image_pos.shape != image_tokens.shape, "负向测试设置失败"

print("✅ 全部检查通过")
print("patches:", patches.shape, patches.dtype)
print("image_tokens:", image_tokens.shape, image_tokens.dtype)
print("text_tokens:", text_tokens.shape, text_tokens.dtype)

✅ 全部检查通过
patches: torch.Size([2, 16, 192]) torch.float32
image_tokens: torch.Size([2, 16, 32]) torch.float32
text_tokens: torch.Size([2, 6, 32]) torch.float32


## Evidence：提交与复现证据

提交 Markdown，固定包含：`patch 数量推导`、`形状数据流`、`核心代码或手算矩阵`、`断言/验证证据`、`位置编码解释`、`投入分钟数`。

### 我的证据

- 代码路径：【填写】
- 配置路径：【填写】
- 数据 / checkpoint / commit 或哈希：【填写】
- 实际命令：【填写】
- 退出码：【填写】
- 关键输出：【填写】
- 结果说明了什么：【填写】
- 结果没有说明什么：【填写】
- 失败现象与定位证据：【填写】

### VLA 约束

| 约束 | 我的定义 |
|---|---|
| 图像布局、颜色顺序和范围 | 【填写】 |
| 文本 token、padding 与 mask | 【填写】 |
| 机器人状态各维含义 | 【填写】 |
| 坐标系、长度和角度单位 | 【填写】 |
| 动作空间及逐维定义 | 【填写】 |
| observation/action 时间对齐 | 【填写】 |
| 控制频率 / action chunk | 【填写】 |
| 归一化及统计量来源 | 【填写】 |
| 随机种子与数据划分 | 【填写】 |


## Self-check：课程自测

1. 为什么文本 id 不能直接和视觉 embedding 做点积？
2. `P` 减半时，图像 token 数变为多少倍？
3. `[CLS]` 会改变哪个维度？
4. 没有位置编码时，注意力对 token 排列有什么性质？


### 自测第 1 题作答

1.因为这两个东西不在同一个空间中，需要先放到统一空间中才能做后续的操作。
2.两个含义不同，不能直接做点积，需要做匹配对应，cross attention等

### 自测第 2 题作答

平方倍

### 自测第 3 题作答

p的维度？

### 自测第 4 题作答

线性？

## Help：排查、最低完成线与提高

### 常见错误

- 直接 `reshape` 导致 patch 内像素交错：先把 patch 网格维与 patch 内维显式拆开，再 `permute`。
- 忘记 RGB 通道，误写 patch 长度 256：应为 `C*P*P`。
- 把 patch 数写成 `224/16=14`：这是每边数量，总数需平方。
- 位置编码维度不匹配：确认其最后两维与 token 序列相同或可广播。

### 最低完成线

手算 224/16 的 196 个 patch，写对三段形状数据流，并完成元素数守恒检查。

### 可选提高

比较 16 与 32 两种 patch 大小的 token 数和注意力矩阵元素数，说明计算量变化趋势。


## Rubric

100 分，80 分通过：patch 计算 15；reshape/permute 逻辑 25；形状与 dtype 25；验证断言 20；位置解释 15。元素数不守恒或将 token id 当 `[B,L,D]` 原始输入，关键项失败。

### 提交前检查

- [ ] 已逐项完成必做作业；
- [ ] 已区分实际结果与预期结果；
- [ ] 已保留代码输出、日志、表格或推理证据；
- [ ] 已记录适用的 shape、坐标系、单位、动作和时间约定；
- [ ] 已完成验证或明确写出无法执行的原因；
- [ ] 已回答全部自测题；
- [ ] 已记录仍不确定的点或失败案例。


## Coach Review（学习者请勿填写）

| 字段 | 验收结果 |
|---|---|
| 证据完整性 | 待验收 |
| Rubric 得分 | /100 |
| 门槛项 | 待验收 |
| 当天状态 | 未提交 |
| 具体缺口 |  |
| 最小补救任务 |  |
| 复验结果 |  |
| 下一课程 |  |


## Next Steps

完成后保存 Notebook，并把路径发到学习对话：

`docs/vla-learning/notebooks/week-01/day-02.ipynb`

教练验收通过后才会更新 `PROGRESS.md`。

[← Week 01 / Day 01](day-01.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-02.md) · [Week 01 / Day 03 →](day-03.ipynb)
